# Proton paper: revision addendum for R2.2, R2.3, and R2.4

**New analyses executed for the revision, not results previously saved in the supplied notebook.**

This standalone notebook reproduces the unchanged three-model 533 kV/cm holdout, performs retrospective whole-field cross-validation, varies the transport coefficient independently on [0,2], evaluates frozen full-energy generators, and computes working-Gaussian information criteria. A new six-parameter biexponential is an explicitly phenomenological comparator.

The embedded numerical statements were extracted verbatim from `Hx_NdNiO3_single_cell_evidence_calibrated_PoC_v3_1(6).ipynb`, whose file and data hashes are retained. No network access is needed. The original notebook is not modified.

**Scope:** The transport sweep is a model-conditional, barrier-raising stress range, not an independently calibrated confidence interval. Its profile audits are frozen-generator checks; nonlinear trajectory/risk ensembles are not rerun. The cross-validation is retrospective, not four new independent experiments. The information criteria assume independent homoscedastic Gaussian errors as a working likelihood; observed temporal correlation limits their evidential interpretation. There is no claim that predictive performance identifies a structural cause.

Dependencies: NumPy, SciPy, pandas. Running the cell writes result tables, per-fold parameters, pointwise predictions and provenance to `proton_review_addendum_results`. No package installation or remote data download is performed.


In [1]:
"""Revision analyses for reviewer comments R2.2--R2.4.

This standalone addendum embeds the selected numerical data and model definitions
from the supplied notebook. It neither requires nor modifies the original notebook.
All new fitting uses only the three training fields for its own fold.
"""
from __future__ import annotations
import ast
import copy
import hashlib
import json
from pathlib import Path
from types import SimpleNamespace
import numpy as np
import pandas as pd
import scipy
from scipy.constants import Boltzmann, atomic_mass, elementary_charge, speed_of_light
from scipy.linalg import eigvalsh
from scipy.optimize import least_squares

SOURCE_NOTEBOOK_NAME = 'Hx_NdNiO3_single_cell_evidence_calibrated_PoC_v3_1(6).ipynb'
SOURCE_NOTEBOOK_SHA256 = '3929eb8def57f4f883706baf2a099b7ad13c8fd6d6bf6cd793ff83afa03640ad'
# This is a verbatim extraction of numerical source statements, not a new data source.
EXTRACTED_SOURCE = 'KB_EV_K = Boltzmann / elementary_charge\n\nEV_A2_TO_N_M = elementary_charge / 1.0e-20\n\nD0_NI_O_A = 1.942\n\nDV_OVER_V_LOCAL = 0.12\n\nNU_CAGE_CM = 420.0\n\nM_O_KG = 15.999 * atomic_mass\n\nOMEGA_CAGE = 2.0 * np.pi * speed_of_light * (NU_CAGE_CM * 100.0)\n\nK_A_EV_A2 = float(M_O_KG * OMEGA_CAGE**2 / EV_A2_TO_N_M)\n\nDELTA_D_A = float(D0_NI_O_A * ((1.0 + DV_OVER_V_LOCAL) ** (1.0 / 3.0) - 1.0))\n\nQ_A_TARGET_A = float(np.sqrt(6.0) * DELTA_D_A)\n\nG_A_EV_A = float(K_A_EV_A2 * Q_A_TARGET_A)\n\nE_A_RELAX_EV = float(0.5 * K_A_EV_A2 * Q_A_TARGET_A**2)\n\nQ_R_BULK_A = 0.06\n\nstrain_pct = np.array([-3.0, 0.0, 3.0])\n\nC_001 = np.array([0.0095, 0.0066, 0.0050])\n\nC_111 = np.array([0.0119, 0.0062, 0.0052])\n\ndef curvature_to_hessian(curvature_mev_per_pct2: np.ndarray, q_bulk_a: float) -> np.ndarray:\n    # If ΔE = C p^2 in meV/f.u. and p=100 q/q_bulk, then 1/2 K q^2 gives K=20 C/q_bulk^2 eV Å^-2.\n    return 20.0 * np.asarray(curvature_mev_per_pct2, dtype=float) / q_bulk_a**2\n\nK_R_001 = curvature_to_hessian(C_001, Q_R_BULK_A)\n\nK_R_111 = curvature_to_hessian(C_111, Q_R_BULK_A)\n\nK_R0_EV_A2 = float(K_R_001[1])\n\nG_R_EV_A = float(K_R0_EV_A2 * Q_R_BULK_A)\n\nE_R_RELAX_EV_FU = float(0.5 * K_R0_EV_A2 * Q_R_BULK_A**2)\n\nD_LOW_M2_S = 6.0e-19\n\nD_HIGH_M2_S = 3.0e-18\n\nD_REP_M2_S = float(np.sqrt(D_LOW_M2_S * D_HIGH_M2_S))\n\nEA_MODEL_EV = 0.41\n\nT_REF_K = 300.0\n\nFIG2H_DATA = np.array([[0.30539, 96.95961, 98.85751, 0.99882, 31.33707, 30.37496, 1.0, 11.25335, 9.67571, 1.0, 3.80501, 2.99641], [0.33516, 97.07396, 97.90469, 1.0, 30.28202, 30.0273, 0.96633, 10.55168, 9.55862, 0.93765, 3.40383, 2.95395], [0.36784, 96.57844, 96.94307, 0.9949, 30.34492, 29.6769, 0.96834, 9.98533, 9.43996, 0.88732, 3.61061, 2.91081], [0.4037, 95.79705, 95.97313, 0.98685, 29.43855, 29.32395, 0.93942, 9.35799, 9.31979, 0.83157, 3.2034, 2.86702], [0.44306, 95.90822, 94.99472, 0.98799, 29.08972, 28.96843, 0.92828, 9.52412, 9.19811, 0.84634, 2.70026, 2.82257], [0.48626, 93.51639, 94.008, 0.96335, 28.02323, 28.61041, 0.89425, 8.72716, 9.07493, 0.77552, 2.7244, 2.77747], [0.53367, 93.49733, 93.0132, 0.96316, 28.28342, 28.25001, 0.90255, 8.84913, 8.95029, 0.78636, 2.45853, 2.73174], [0.5857, 91.46444, 92.01044, 0.94221, 27.73445, 27.88728, 0.88504, 8.23672, 8.82423, 0.73194, 2.51666, 2.6854], [0.64281, 91.27068, 90.9997, 0.94022, 26.98534, 27.52225, 0.86113, 8.47432, 8.69675, 0.75305, 2.41946, 2.63846], [0.70548, 89.67613, 89.98138, 0.92379, 26.66224, 27.15509, 0.85082, 8.40475, 8.56792, 0.74687, 2.16408, 2.59094], [0.77426, 89.13296, 88.95552, 0.9182, 26.82522, 26.78583, 0.85602, 8.06806, 8.43775, 0.71695, 2.32258, 2.54287], [0.84975, 87.87193, 87.92222, 0.90521, 25.72156, 26.41454, 0.8208, 7.55475, 8.30627, 0.67133, 2.10564, 2.49424], [0.9326, 87.27159, 86.88171, 0.89902, 25.50426, 26.04133, 0.81387, 7.77614, 8.17354, 0.69101, 2.17774, 2.4451], [1.02353, 85.44516, 85.83415, 0.88021, 25.27552, 25.66628, 0.80657, 7.79044, 8.03958, 0.69228, 2.29082, 2.39547], [1.12332, 84.35566, 84.77982, 0.86898, 25.1297, 25.28952, 0.80192, 7.45723, 7.90446, 0.66267, 1.89536, 2.34537], [1.23285, 84.42554, 83.71875, 0.8697, 25.06394, 24.9111, 0.79982, 7.37274, 7.76821, 0.65516, 2.13359, 2.29482], [1.35305, 83.45991, 82.65135, 0.85976, 24.24906, 24.53117, 0.77381, 7.01349, 7.6309, 0.62324, 2.26763, 2.24387], [1.48497, 82.25606, 81.57774, 0.84735, 24.36915, 24.14982, 0.77765, 7.40387, 7.49257, 0.65793, 1.99669, 2.19254], [1.62975, 80.75044, 80.49816, 0.83184, 23.56857, 23.76716, 0.7521, 6.983, 7.35328, 0.62053, 1.68572, 2.14086], [1.78865, 79.72764, 79.41283, 0.82131, 23.04247, 23.38329, 0.73531, 6.96743, 7.21309, 0.61914, 2.08912, 2.08887], [1.96304, 77.57087, 78.32202, 0.79909, 22.47348, 22.99834, 0.71715, 7.17993, 7.07206, 0.63803, 2.04941, 2.03661], [2.15443, 78.1045, 77.226, 0.80459, 22.28763, 22.61244, 0.71122, 6.65615, 6.93027, 0.59148, 2.09484, 1.98411], [2.36449, 76.96417, 76.12496, 0.79284, 22.71366, 22.22568, 0.72482, 6.71745, 6.78777, 0.59693, 1.92966, 1.93142], [2.59502, 75.17586, 75.01928, 0.77442, 22.13324, 21.83823, 0.7063, 6.77971, 6.64465, 0.60246, 1.69334, 1.87858], [2.84804, 73.00003, 73.90913, 0.752, 21.68434, 21.45017, 0.69197, 6.69236, 6.50097, 0.5947, 1.66348, 1.82563], [3.12572, 74.29917, 72.79492, 0.76539, 21.19827, 21.06169, 0.67646, 6.4789, 6.35682, 0.57573, 1.34902, 1.77262], [3.43047, 70.83372, 71.6769, 0.72969, 20.45773, 20.67289, 0.65283, 5.67559, 6.21228, 0.50435, 1.5936, 1.71959], [3.76494, 71.93276, 70.55536, 0.74101, 20.99527, 20.28391, 0.66998, 5.63779, 6.06743, 0.50099, 1.66126, 1.66659], [4.13201, 71.42135, 69.43068, 0.73574, 19.254, 19.89492, 0.61442, 5.46277, 5.92235, 0.48544, 1.20957, 1.61368], [4.97702, 67.98131, 67.17309, 0.7003, 18.4477, 19.11741, 0.58869, 4.96408, 5.63189, 0.44112, 1.94364, 1.50832], [5.46228, 64.81444, 66.04087, 0.66768, 18.62211, 18.72919, 0.59425, 6.27212, 5.48669, 0.55736, 1.95762, 1.45597], [5.99484, 66.67264, 64.90688, 0.68682, 19.33692, 18.34155, 0.61706, 5.39226, 5.34163, 0.47917, 1.02248, 1.40392], [6.57933, 62.20979, 63.77144, 0.64085, 18.73648, 17.95463, 0.5979, 5.83727, 5.19682, 0.51871, 1.98906, 1.35221], [7.22081, 64.64292, 62.63494, 0.66591, 18.08458, 17.56858, 0.5771, 5.90779, 5.05236, 0.52498, 1.9465, 1.30092], [7.92483, 60.09114, 61.49776, 0.61902, 18.62783, 17.18357, 0.59443, 4.78556, 4.90835, 0.42526, 1.23689, 1.25008], [8.69749, 62.02874, 60.36031, 0.63898, 17.09243, 16.79976, 0.54544, 5.51455, 4.76489, 0.49004, 1.87725, 1.19977], [9.54548, 58.49341, 59.22297, 0.60257, 15.99735, 16.41731, 0.51049, 4.94978, 4.62209, 0.43985, 0.87605, 1.15003], [10.47616, 58.55376, 58.08613, 0.60319, 15.72286, 16.03638, 0.50173, 3.92476, 4.48005, 0.34876, 0.67816, 1.10092], [11.49757, 58.48388, 56.95024, 0.60247, 16.06025, 15.65715, 0.5125, 5.19786, 4.33889, 0.46189, 1.43065, 1.0525], [12.61857, 54.37362, 55.81569, 0.56013, 15.84867, 15.27977, 0.50575, 4.08231, 4.19872, 0.36276, 1.18829, 1.00482], [13.84886, 53.84951, 54.68292, 0.55473, 14.51913, 14.90442, 0.46332, 4.62516, 4.05963, 0.411, 1.20417, 0.95794], [15.19911, 53.53822, 53.55236, 0.55152, 14.91084, 14.53126, 0.47582, 4.30593, 3.92175, 0.38264, 0.78489, 0.91191], [16.68101, 51.79756, 52.42443, 0.53359, 14.42191, 14.16045, 0.46022, 3.92857, 3.78518, 0.3491, 0.82809, 0.86678], [18.30738, 51.43545, 51.29961, 0.52986, 14.14743, 13.79218, 0.45146, 3.83423, 3.65003, 0.34072, 1.11142, 0.8226], [20.09233, 49.92983, 50.17831, 0.51435, 13.15242, 13.42661, 0.41971, 3.46005, 3.51641, 0.30747, 0.88844, 0.77943], [22.05131, 47.83976, 49.061, 0.49282, 13.45264, 13.0639, 0.42929, 3.73036, 3.38442, 0.33149, 1.4027, 0.73729], [24.20128, 46.26427, 47.94813, 0.47659, 13.76143, 12.70422, 0.43914, 3.06554, 3.25419, 0.27241, 1.47194, 0.69625], [26.56088, 47.80482, 46.84017, 0.49246, 11.87435, 12.34774, 0.37892, 2.69549, 3.1258, 0.23953, 0.21123, 0.65634], [29.15053, 44.70465, 45.73757, 0.46052, 11.37113, 11.99462, 0.36286, 2.66881, 2.99937, 0.23716, 0.66673, 0.6176], [31.99267, 44.31713, 44.64079, 0.45653, 11.41401, 11.64504, 0.36423, 3.01313, 2.87498, 0.26775, 0.56858, 0.58006], [35.11192, 42.77023, 43.55031, 0.44059, 11.13953, 11.29914, 0.35547, 2.78316, 2.75276, 0.24732, 0.75884, 0.54376], [38.53529, 42.40494, 42.46659, 0.43683, 10.42472, 10.95709, 0.33266, 2.22062, 2.63278, 0.19733, 1.11396, 0.50872], [42.29243, 41.16932, 41.3901, 0.4241, 9.97011, 10.61904, 0.31816, 3.17672, 2.51514, 0.28229, 0.09116, 0.47497], [46.41589, 40.23546, 40.32129, 0.41448, 10.29034, 10.28516, 0.32838, 2.3931, 2.39992, 0.21266, 0.06067, 0.44252], [50.94138, 38.49162, 39.26065, 0.39652, 9.69562, 9.95559, 0.3094, 2.29558, 2.28723, 0.20399, 0.37577, 0.4114], [55.9081, 36.81448, 38.20862, 0.37924, 9.94723, 9.63048, 0.31743, 2.22539, 2.17712, 0.19775, 0.38561, 0.38161], [61.35907, 36.86212, 37.16568, 0.37973, 8.47759, 9.30997, 0.27053, 2.46235, 2.06968, 0.21881, -0.18487, 0.35316], [67.34151, 35.88697, 36.13228, 0.36969, 8.65772, 8.99421, 0.27628, 1.98461, 1.96498, 0.17636, 0.44628, 0.32606], [73.90722, 34.76252, 35.10888, 0.3581, 7.78566, 8.68333, 0.24845, 2.5624, 1.86308, 0.2277, 0.5943, 0.3003], [81.11308, 33.35538, 34.09591, 0.34361, 8.16022, 8.37747, 0.2604, 1.84644, 1.76405, 0.16408, 0.48726, 0.27588], [89.02151, 33.46337, 33.09384, 0.34472, 8.69203, 8.07675, 0.27737, 1.44907, 1.66793, 0.12877, 0.07877, 0.25279], [97.701, 32.06258, 32.10308, 0.33029, 7.47687, 7.78129, 0.23859, 1.59265, 1.57477, 0.14153, -0.00794, 0.23101], [107.22672, 29.70569, 31.12408, 0.30601, 6.54762, 7.4912, 0.20894, 1.34584, 1.48462, 0.11959, -0.41134, 0.21052], [117.6812, 30.49026, 30.15726, 0.31409, 7.34248, 7.20661, 0.23431, 1.27977, 1.3975, 0.11372, 0.12102, 0.19131], [129.15497, 29.20382, 29.20303, 0.30084, 6.44469, 6.92761, 0.20566, 1.81182, 1.31344, 0.161, -0.21917, 0.17334], [141.74742, 27.52668, 28.26179, 0.28356, 6.65055, 6.6543, 0.21223, 1.31217, 1.23248, 0.1166, 0.25634, 0.15658], [155.56761, 28.53043, 27.33393, 0.2939, 6.59337, 6.38678, 0.2104, 0.97738, 1.1546, 0.08685, -0.32304, 0.141], [170.73526, 25.77013, 26.41985, 0.26547, 6.68486, 6.12512, 0.21332, 0.87923, 1.07983, 0.07813, 0.0162, 0.12657], [187.38174, 28.29537, 25.5199, 0.29148, 5.32101, 5.8694, 0.1698, 0.68674, 1.00817, 0.06103, -0.36973, 0.11323], [205.65123, 26.00201, 24.63444, 0.26786, 6.34461, 5.6197, 0.20246, 0.35322, 0.93959, 0.03139, -0.26841, 0.10096], [225.70197, 23.0289, 23.76382, 0.23723, 5.23809, 5.37608, 0.16715, 1.05202, 0.87409, 0.09349, 0.53745, 0.0897], [247.70764, 22.90184, 22.90837, 0.23592, 5.7213, 5.13859, 0.18257, 0.69277, 0.81165, 0.06156, -0.14167, 0.07941], [271.85882, 21.67575, 22.0684, 0.22329, 5.2095, 4.90727, 0.16624, 0.86843, 0.75223, 0.07717, -0.52824, 0.07003], [298.36472, 21.45658, 21.2442, 0.22103, 4.80349, 4.68218, 0.15328, 0.83253, 0.69579, 0.07398, -0.13309, 0.06153], [327.45492, 20.4179, 20.43606, 0.21033, 5.18091, 4.46333, 0.16533, 0.31542, 0.6423, 0.02803, 0.39769, 0.05384], [359.38137, 19.17593, 19.64424, 0.19754, 4.33458, 4.25076, 0.13832, -0.09275, 0.59169, -0.00824, -0.46884, 0.04693], [394.42061, 18.59147, 18.86899, 0.19152, 3.92285, 4.04446, 0.12518, 0.135, 0.54393, 0.012, 0.05178, 0.04072], [432.87613, 18.91546, 18.11052, 0.19486, 3.99147, 3.84447, 0.12737, 0.55269, 0.49893, 0.04911, 0.1045, 0.03519], [475.08102, 17.49561, 17.36905, 0.18023, 3.44822, 3.65075, 0.11004, -0.04098, 0.45665, -0.00364, -0.35544, 0.03027], [521.40083, 17.05092, 16.64477, 0.17565, 3.19947, 3.46332, 0.1021, 0.47519, 0.41699, 0.04223, 0.53205, 0.02592], [572.23677, 17.34632, 15.93784, 0.17869, 2.81347, 3.28215, 0.08978, 0.45105, 0.37989, 0.04008, -0.39959, 0.02209], [628.02914, 16.16788, 15.24841, 0.16655, 3.11655, 3.1072, 0.09945, -0.16613, 0.34525, -0.01476, -0.34337, 0.01873], [689.26121, 13.95393, 14.57659, 0.14375, 3.00504, 2.93845, 0.09589, 0.10514, 0.31301, 0.00934, -0.63719, 0.0158], [756.46333, 13.66487, 13.9225, 0.14077, 2.15585, 2.77585, 0.0688, -0.19471, 0.28306, -0.0173, 0.62956, 0.01327], [830.21757, 13.15347, 13.28621, 0.1355, 3.20519, 2.61934, 0.10228, -0.00858, 0.25531, -0.000762109, 0.17661, 0.01107], [911.16276, 13.69664, 12.66778, 0.14109, 3.148, 2.46887, 0.10046, 0.22044, 0.22967, 0.01959, 0.04733, 0.0092], [1000.0, 11.98456, 12.06725, 0.12346, 2.39317, 2.32438, 0.07637, 0.20647, 0.20604, 0.01835, 0.00604, 0.00759]], dtype=float)\n\nEXP_TIME_S = FIG2H_DATA[:, 0] * 1.0e-3\n\nEXP_FIELDS_KV_CM = np.array([133.0, 267.0, 400.0, 533.0])\n\nEXP_Y = np.column_stack([\n    FIG2H_DATA[:, 10] / FIG2H_DATA[0, 10],\n    FIG2H_DATA[:, 9],\n    FIG2H_DATA[:, 6],\n    FIG2H_DATA[:, 3],\n])\n\nEXP_Y = np.clip(EXP_Y, 0.0, 1.2)\n\nCAL_FIELD_IDX = np.array([0, 1, 2], dtype=int)\n\nHOLDOUT_FIELD_IDX = 3\n\nFIELD_SCALED = (EXP_FIELDS_KV_CM - 333.0) / 200.0\n\ndef experimental_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, float]:\n    y_true = np.asarray(y_true, dtype=float)\n    y_pred = np.asarray(y_pred, dtype=float)\n    residual = y_pred - y_true\n    rmse = float(np.sqrt(np.mean(residual**2)))\n    value_range = max(float(np.ptp(y_true)), 1e-15)\n    sst = max(float(np.sum((y_true - y_true.mean()) ** 2)), 1e-15)\n    lag1 = float(np.corrcoef(residual[:-1], residual[1:])[0, 1]) if len(residual) > 2 else float("nan")\n    dw = float(np.sum(np.diff(residual) ** 2) / max(np.sum(residual**2), 1e-15))\n    return {\n        "RMSE": rmse,\n        "NRMSE/range": rmse / value_range,\n        "R2": 1.0 - float(np.sum(residual**2)) / sst,\n        "MAE": float(np.mean(np.abs(residual))),\n        "lag-1 residual correlation": lag1,\n        "Durbin-Watson": dw,\n    }\n\ndef single_exp_prediction(theta: np.ndarray, field_idx: int) -> np.ndarray:\n    tau = np.exp(theta[0] + theta[1] * FIELD_SCALED[field_idx])\n    return np.exp(-EXP_TIME_S / tau)\n\ndef single_exp_residual(theta: np.ndarray) -> np.ndarray:\n    return np.concatenate([single_exp_prediction(theta, i) - EXP_Y[:, i] for i in CAL_FIELD_IDX])\n\nSINGLE_FIT = least_squares(\n    single_exp_residual,\n    x0=np.array([np.log(0.01), 0.5]),\n    bounds=(np.array([-20.0, -10.0]), np.array([5.0, 10.0])),\n    max_nfev=10000,\n)\n\ndef logistic_scalar(value: float) -> float:\n    return float(1.0 / (1.0 + np.exp(-np.clip(value, -50.0, 50.0))))\n\ndef stretched_prediction(theta: np.ndarray, field_idx: int) -> np.ndarray:\n    scaled_field = FIELD_SCALED[field_idx]\n    tau = np.exp(theta[0] + theta[1] * scaled_field)\n    beta = 0.05 + 0.90 * logistic_scalar(theta[2] + theta[3] * scaled_field)\n    return np.exp(-np.power(EXP_TIME_S / tau, beta))\n\ndef stretched_residual(theta: np.ndarray) -> np.ndarray:\n    return np.concatenate([stretched_prediction(theta, i) - EXP_Y[:, i] for i in CAL_FIELD_IDX])\n\nSTRETCHED_FIT = least_squares(\n    stretched_residual,\n    x0=np.array([np.log(0.005), 0.5, -1.0, -0.5]),\n    bounds=(np.array([-20.0, -10.0, -10.0, -10.0]), np.array([5.0, 10.0, 10.0, 10.0])),\n    max_nfev=15000,\n)\n\nTRAP_Z = np.linspace(0.0, 1.0, 21)\n\nTRAP_T_K = 300.0\n\nTRAP_BOUNDS = (\n    np.array([np.log(1e-6), 0.05, -20.0, -20.0, -20.0, -5.0]),\n    np.array([np.log(1.0), 3.00, 20.0, 20.0, 20.0, 5.0]),\n)\n\ndef trap_prediction(theta: np.ndarray, field_idx: int) -> np.ndarray:\n    log_tau_base, chi_a, b0, b1, b2, c_field = np.asarray(theta, dtype=float)\n    scaled_field = FIELD_SCALED[field_idx]\n    log_tau = (\n        log_tau_base\n        + chi_a * E_A_RELAX_EV / (KB_EV_K * TRAP_T_K) * TRAP_Z\n        + c_field * scaled_field\n    )\n    tau = np.exp(np.clip(log_tau, -50.0, 50.0))\n    logits = (b0 + b1 * scaled_field) * TRAP_Z + b2 * (TRAP_Z - 0.5) ** 2\n    logits -= logits.max()\n    weights = np.exp(logits)\n    weights /= weights.sum()\n    return np.exp(-EXP_TIME_S[:, None] / tau[None, :]) @ weights\n\ndef trap_residual(theta: np.ndarray, observed: np.ndarray = EXP_Y) -> np.ndarray:\n    return np.concatenate([trap_prediction(theta, i) - observed[:, i] for i in CAL_FIELD_IDX])\n\nTRAP_FIT = least_squares(\n    trap_residual,\n    x0=np.array([np.log(7.0e-4), 1.0, 0.0, 0.0, 0.0, 1.0]),\n    bounds=TRAP_BOUNDS,\n    max_nfev=20000,\n    xtol=1e-11,\n    ftol=1e-11,\n    gtol=1e-11,\n)\n\ndef coupled_local_generator(\n    profile: np.ndarray,\n    temperature_k: float,\n    field_v_m: float,\n    strain: float = 0.0,\n) -> tuple[np.ndarray, np.ndarray, np.ndarray, float, list[tuple[int, int, float, float, float]]]:\n    """Mean-field local detailed-balance generator with explicit structural feedback.\n\n    The local donated-electron occupancy is n_e=c and Q_A,i=Q_A^(1)n_e,i.\n    The edge mobility is reduced by the fitted structural barrier transfer chi_A.\n    A bounded staggered R2 mode and strain shift are included without allowing\n    recoverability to enter the rates or forces.\n    """\n    c = np.clip(np.asarray(profile, dtype=float), 1e-12, 1.0 - 1e-12)\n    n = len(c)\n    h = CFG.length_m / n\n    x = (np.arange(n) + 0.5) * h\n    n_e = c.copy()\n    q_a = Q_A_TARGET_A * n_e\n    alternating = (-1.0) ** np.arange(n)\n    m_pi = float(abs(alternating @ n_e) / max(n_e.sum(), 1e-15))\n    q_r = float(Q_R_BULK_A * m_pi)\n\n    d_reference = D_REP_M2_S * np.exp(-EA_MODEL_EV / KB_EV_K * (1.0 / temperature_k - 1.0 / T_REF_K))\n    beta = 1.0 / (KB_EV_K * temperature_k)\n    # A weak site-energy contribution represents local electron-lattice trapping.\n    u_ev = (\n        -field_v_m * x\n        - 0.20 * E_A_RELAX_EV * (q_a / Q_A_TARGET_A)\n        - 0.10 * E_R_RELAX_EV_FU * (q_r / Q_R_BULK_A if q_r > 0.0 else 0.0) * alternating\n    )\n    q_matrix = np.zeros((n, n), dtype=float)\n    edge_records = []\n    for i in range(n - 1):\n        j = i + 1\n        q_edge_fraction = 0.5 * (q_a[i] + q_a[j]) / Q_A_TARGET_A\n        barrier_shift_ev = (\n            TRAP_FIT.x[1] * E_A_RELAX_EV * q_edge_fraction\n            + 0.20 * E_R_RELAX_EV_FU * (q_r / Q_R_BULK_A if q_r > 0.0 else 0.0)\n            + 0.15 * E_A_RELAX_EV * strain / 0.01\n        )\n        edge_diffusivity = d_reference * np.exp(-barrier_shift_ev * beta)\n        k0 = edge_diffusivity / h**2\n        du = u_ev[j] - u_ev[i]\n        k_i_j = k0 * np.exp(-0.5 * beta * du)\n        k_j_i = k0 * np.exp(+0.5 * beta * du)\n        q_matrix[j, i] += k_i_j\n        q_matrix[i, i] -= k_i_j\n        q_matrix[i, j] += k_j_i\n        q_matrix[j, j] -= k_j_i\n        edge_records.append((i, j, k_i_j, k_j_i, du))\n    return q_matrix, q_a, u_ev, q_r, edge_records'
OUT = Path(globals().get("__file__", "proton_review_addendum.py")).resolve().parent / "proton_review_addendum_results"
OUT.mkdir(parents=True, exist_ok=True)
source = EXTRACTED_SOURCE
tree = ast.parse(source)
NS = dict(np=np, Boltzmann=Boltzmann, atomic_mass=atomic_mass,
          elementary_charge=elementary_charge, speed_of_light=speed_of_light,
          least_squares=least_squares)
assign_names = {
 'KB_EV_K', 'EV_A2_TO_N_M', 'D0_NI_O_A', 'DV_OVER_V_LOCAL', 'NU_CAGE_CM',
 'M_O_KG', 'OMEGA_CAGE', 'K_A_EV_A2', 'DELTA_D_A', 'Q_A_TARGET_A',
 'G_A_EV_A', 'E_A_RELAX_EV', 'Q_R_BULK_A', 'strain_pct', 'C_001', 'C_111',
 'K_R_001', 'K_R_111', 'K_R0_EV_A2', 'G_R_EV_A', 'E_R_RELAX_EV_FU',
 'D_LOW_M2_S', 'D_HIGH_M2_S', 'D_REP_M2_S', 'EA_MODEL_EV', 'T_REF_K',
 'FIG2H_DATA', 'EXP_TIME_S', 'EXP_FIELDS_KV_CM', 'EXP_Y', 'CAL_FIELD_IDX',
 'HOLDOUT_FIELD_IDX', 'FIELD_SCALED', 'TRAP_Z', 'TRAP_T_K', 'TRAP_BOUNDS'
}
func_names = {'curvature_to_hessian','experimental_metrics', 'single_exp_prediction',
 'single_exp_residual','logistic_scalar','stretched_prediction','stretched_residual',
 'trap_prediction','trap_residual','coupled_local_generator'}
fit_nodes = {}
for node in tree.body:
    if isinstance(node, ast.Assign):
        names={t.id for t in node.targets if isinstance(t, ast.Name)}
        if names & {'SINGLE_FIT', 'STRETCHED_FIT', 'TRAP_FIT'}:
            fit_nodes[next(iter(names))]=node
        if names & assign_names:
            exec(compile(ast.Module(body=[node], type_ignores=[]),SOURCE_NOTEBOOK_NAME,'exec'),NS)
    elif isinstance(node,ast.FunctionDef) and node.name in func_names:
        exec(compile(ast.Module(body=[node],type_ignores=[]),SOURCE_NOTEBOOK_NAME,'exec'),NS)
# Give the independent transport sensitivity an explicit keyword argument.
original_generator = next(n for n in tree.body if isinstance(n, ast.FunctionDef) and n.name == 'coupled_local_generator')
independent_generator = copy.deepcopy(original_generator)
independent_generator.name = 'independent_local_generator'
independent_generator.args.kwonlyargs.append(ast.arg(arg='chi_A_tr'))
independent_generator.args.kw_defaults.append(None)
class IndependentCoefficient(ast.NodeTransformer):
    def visit_Subscript(self, node):
        if ast.unparse(node) == 'TRAP_FIT.x[1]':
            return ast.copy_location(ast.Name(id='chi_A_tr', ctx=ast.Load()), node)
        return self.generic_visit(node)
independent_generator = IndependentCoefficient().visit(independent_generator)
ast.fix_missing_locations(independent_generator)
exec(compile(ast.Module(body=[independent_generator], type_ignores=[]), '<independent coefficient>', 'exec'), NS)
NS['CFG']=SimpleNamespace(n_cells=32,length_m=100e-9,temperature_k=300.,field_v_m=2e5)
assert NS['EXP_Y'].shape==(87,4)
models={'Single exponential':('SINGLE_FIT','single_exp_prediction'),
        'Stretched exponential':('STRETCHED_FIT','stretched_prediction'),
        'Distributed relaxation':('TRAP_FIT','trap_prediction')}
all_rows=[]; fits={}; predictions=[]
# Preserve the original optimizer starts, bounds, tolerances and budgets.
for fold in range(4):
    NS['CAL_FIELD_IDX']=np.array([i for i in range(4) if i != fold])
    for label,(fitname,predname) in models.items():
        exec(compile(ast.Module(body=[fit_nodes[fitname]],type_ignores=[]),SOURCE_NOTEBOOK_NAME,'exec'),NS)
        fit=NS[fitname]; pred=NS[predname](fit.x,fold)
        assert fit.success and np.isfinite(pred).all()
        fits[(fold,label)]=fit
        r=NS['experimental_metrics'](NS['EXP_Y'][:,fold],pred)
        all_rows.append(dict(field_kV_cm=NS['EXP_FIELDS_KV_CM'][fold],model=label,
             parameters=len(fit.x),train_sse=np.dot(fit.fun,fit.fun),train_n=len(fit.fun),
             train_rmse=np.sqrt(np.mean(fit.fun**2)),**r,nfev=fit.nfev,
             optimality=fit.optimality,status=fit.status))
        predictions.extend(dict(field_kV_cm=NS['EXP_FIELDS_KV_CM'][fold],model=label,
                    time_s=t,observed=y,predicted=p) for t,y,p in zip(NS['EXP_TIME_S'],NS['EXP_Y'][:,fold],pred))
        print(f'fold={fold} {label}: train={np.sqrt(np.mean(fit.fun**2)):.9g} test={r["RMSE"]:.9g} nfev={fit.nfev}',flush=True)

# Additional comparator: six freely fitted parameters, no structural energy.
# Three fixed initial time-scale pairs are applied to every fold, and the winner
# is selected by TRAINING residual sum of squares only. No held-out prediction
# participates in optimization or start selection.
t=NS['EXP_TIME_S']; F=NS['FIELD_SCALED']; Y=NS['EXP_Y']
def biexponential(theta,field):
    a0,a1,d0,d1,g0,g1=np.asarray(theta)
    tau1=np.exp(a0+a1*F[field]); tau2=np.exp(d0+d1*F[field])
    w=1/(1+np.exp(-(g0+g1*F[field])))
    return w*np.exp(-t/tau1)+(1-w)*np.exp(-t/tau2)
BI_BOUNDS=(np.array([-20.,-10.,-20.,-10.,-10.,-10.]),np.array([5.,10.,5.,10.,10.,10.]))
BI_STARTS=[np.array([np.log(ts),.5,np.log(tl),.5,0.,0.]) for ts,tl in [(1e-4,.1),(1e-3,1.),(1e-2,10.)]]
bi_start_records=[]
for fold in range(4):
    train=[i for i in range(4) if i != fold]
    def residual(theta):
        return np.concatenate([biexponential(theta,i)-Y[:,i] for i in train])
    candidates=[least_squares(residual,x,bounds=BI_BOUNDS,max_nfev=20000,
                   xtol=1e-11,ftol=1e-11,gtol=1e-11) for x in BI_STARTS]
    assert all(f.success for f in candidates)
    fit=min(candidates,key=lambda f:np.dot(f.fun,f.fun)); label='Biexponential (six parameters)'
    fits[(fold,label)]=fit
    for ix,cf in enumerate(candidates):
        bi_start_records.append(dict(field_kV_cm=NS['EXP_FIELDS_KV_CM'][fold],start=ix,
                   train_sse=np.dot(cf.fun,cf.fun),success=bool(cf.success),theta=cf.x.tolist()))
    pred=biexponential(fit.x,fold);r=NS['experimental_metrics'](Y[:,fold],pred)
    all_rows.append(dict(field_kV_cm=NS['EXP_FIELDS_KV_CM'][fold],model=label,
             parameters=6,train_sse=np.dot(fit.fun,fit.fun),train_n=len(fit.fun),
             train_rmse=np.sqrt(np.mean(fit.fun**2)),**r,nfev=fit.nfev,optimality=fit.optimality,status=fit.status))
    predictions.extend(dict(field_kV_cm=NS['EXP_FIELDS_KV_CM'][fold],model=label,
                    time_s=tm,observed=y,predicted=p) for tm,y,p in zip(t,Y[:,fold],pred))
    print(f'fold={fold} {label}: train={np.sqrt(np.mean(fit.fun**2)):.9g} test={r["RMSE"]:.9g} nfev={fit.nfev}',flush=True)
cv=pd.DataFrame(all_rows);cv.to_csv(OUT/'leave_one_field_out.csv',index=False)
pd.DataFrame(predictions).to_csv(OUT/'out_of_fold_predictions.csv',index=False)
summary=[]
for model,g in cv.groupby('model',sort=False):
    summary.append(dict(model=model,mean_field_rmse=g['RMSE'].mean(),
                pooled_rmse=np.sqrt(np.mean(g['RMSE']**2)),
                interpolation_rmse=np.sqrt(np.mean(g[g.field_kV_cm.isin([267.,400.])]['RMSE']**2)),
                extrapolation_rmse=np.sqrt(np.mean(g[g.field_kV_cm.isin([133.,533.])]['RMSE']**2))))
pd.DataFrame(summary).to_csv(OUT/'cross_validation_summary.csv',index=False)

# Independent transport-coefficient sensitivity. Only the explicit coefficient
# argument changes. No relaxation parameters are modified or re-estimated.
basefit=fits[(3,'Distributed relaxation')]
chi0=float(basefit.x[1]);kb=NS['KB_EV_K'];EA=NS['E_A_RELAX_EV'];T=300.;L=1e-7
chi_values=[0.,.25,.5,chi0,1.,1.5,2.]
N=32;x=(np.arange(N)+.5)/N
frozen_profiles={'uniform_0.1':np.full(N,.1),'uniform_0.5':np.full(N,.5),
    'nonuniform_staggered':.3+.12*np.cos(2*np.pi*x)+.05*(-1.)**np.arange(N)}
sens=[]; aud=[]
for chi in chi_values:
    D=NS['D_REP_M2_S']*np.exp(-chi*EA*.3/(kb*T))
    gap_lower=D*np.pi**2/L**2*np.exp(-2e5*L/(kb*T))
    qs={}
    for name,c in frozen_profiles.items():
        q,qa,u,qr,edges=NS['independent_local_generator'](c,T,2e5,0.,chi_A_tr=chi)
        pi=np.exp(-(u-u.min())/(kb*T));pi/=pi.sum()
        sym=q*np.sqrt(pi)[None,:]/np.sqrt(pi)[:,None]
        gap=-eigvalsh((sym+sym.T)/2)[-2]
        p=.15+.7*np.exp(-((x-.35)/.18)**2);p/=p.sum()
        dent=float((q@p)@np.log(p/pi))
        rates=np.array([[e[2],e[3]] for e in edges])
        scale=max(np.max(np.abs(q)),1e-300)
        aud.append(dict(chi=chi,profile=name,Q_R_A=qr,
             frozen_gap_s_1=gap,entropy_derivative_s_1=dent,
             detailed_balance_log_error=max(abs(np.log(a/b)+du/(kb*T)) for _,_,a,b,du in edges),
             relative_column_sum_error=np.max(np.abs(q.sum(0)))/scale,
             relative_stationarity_error=np.linalg.norm(q@pi)/scale,
             min_rate_s_1=rates.min(),max_rate_s_1=rates.max()))
        qs[name]=q
    effect=np.linalg.norm(qs['uniform_0.5']-qs['uniform_0.1'],2)/np.linalg.norm(qs['uniform_0.1'],2)
    formula=1-np.exp(-chi*EA*.4/(kb*T))
    assert np.isclose(effect,formula,atol=1e-13)
    sens.append(dict(chi=chi,barrier_at_cbar_eV=chi*EA*.3,D_eff_m2_s=D,
        diffusion_time_s=L**2/D,gap_lower_s_1=gap_lower,operator_change=effect))
NS['TRAP_FIT']=basefit
for c in frozen_profiles.values():
    q0=NS['coupled_local_generator'](c,T,2e5,0.)[0]
    q1=NS['independent_local_generator'](c,T,2e5,0.,chi_A_tr=chi0)[0]
    assert np.array_equal(q0,q1), 'The independent-coefficient generator must reproduce the original at nominal chi.'
pd.DataFrame(sens).to_csv(OUT/'transport_sensitivity.csv',index=False)
pd.DataFrame(aud).to_csv(OUT/'frozen_generator_audits.csv',index=False)
# Criteria at the unchanged original calibration partition only; sigma^2 is
# counted as one fitted likelihood parameter. Common Gaussian constants omitted.
ics=[]
for label,g in cv[cv.field_kV_cm==533.].groupby('model',sort=False):
    row=g.iloc[0];n=int(row.train_n);k=int(row.parameters)+1;rss=row.train_sse
    dev=n*np.log(rss/n)
    ics.append(dict(model=label,mean_parameters=k-1,likelihood_parameters=k,
        n=n,sse=rss,AIC=dev+2*k,AICc=dev+2*k+2*k*(k+1)/(n-k-1),BIC=dev+k*np.log(n)))
ic=pd.DataFrame(ics)
for col in ['AIC','AICc','BIC']:ic['delta_'+col]=ic[col]-ic[col].min()
ic.to_csv(OUT/'information_criteria.csv',index=False)
params=[dict(field_kV_cm=float(NS['EXP_FIELDS_KV_CM'][fold]),model=model,
             theta=fit.x.tolist(),success=bool(fit.success),message=fit.message,
             active_mask=fit.active_mask.tolist(),nfev=int(fit.nfev),optimality=float(fit.optimality))
        for (fold,model),fit in fits.items()]
provenance=dict(source_notebook=SOURCE_NOTEBOOK_NAME,source_sha256=SOURCE_NOTEBOOK_SHA256,
   embedded_matrix_sha256=hashlib.sha256(np.ascontiguousarray(NS['FIG2H_DATA']).tobytes()).hexdigest(),
   numpy_version=np.__version__,scipy_version=scipy.__version__,chi_reference=chi0,
   E_A_eV=EA,k_B_eV_K=kb,D_rep_m2_s=NS['D_REP_M2_S'],
   analyses_are_new_revision_addendum=True,original_models_changed=False,
   optimizer_rule='Original single starts/settings; biexponential uses three fixed starts chosen by training RSS.',
   information_criteria='Working independent homoscedastic Gaussian likelihood; shared fitted variance counted; constants omitted; temporal correlation not modelled.',
   cross_validation_scope='Retrospective whole-field refits; original 533 holdout not re-designated as four independent experiments.',
   sensitivity_scope='Positive-barrier coefficient sector [0,2], user-model conditional stress range; not an empirical confidence interval. Frozen generators and smooth constant-D scaling only; risk ensembles not rerun.')
(OUT/'fit_parameters.json').write_text(json.dumps(params,indent=2))
(OUT/'biexponential_start_audit.json').write_text(json.dumps(bi_start_records,indent=2))
(OUT/'provenance.json').write_text(json.dumps(provenance,indent=2))
print('\nCROSS VALIDATION\n',cv[['field_kV_cm','model','RMSE','R2']].to_string(index=False))
print('\nSUMMARY\n',pd.DataFrame(summary).to_string(index=False))
print('\nCRITERIA\n',ic.to_string(index=False))
print('\nSENSITIVITY\n',pd.DataFrame(sens).to_string(index=False))
print('\nPROVENANCE\n',json.dumps(provenance,indent=2))

# Numerical start sensitivity audit for the three inherited models. The original
# fits and original held-out predictions are retained irrespective of this audit.
start_audit=[]
for fold in range(4):
    train=[i for i in range(4) if i != fold]
    definitions=[
      ('Single exponential','single_exp_prediction',np.array([np.log(.01),.5]),(np.array([-20.,-10.]),np.array([5.,10.]))),
      ('Stretched exponential','stretched_prediction',np.array([np.log(.005),.5,-1.,-.5]),(np.array([-20.,-10.,-10.,-10.]),np.array([5.,10.,10.,10.]))),
      ('Distributed relaxation','trap_prediction',np.array([np.log(7e-4),1.,0.,0.,0.,1.]),NS['TRAP_BOUNDS'])]
    for label,predname,x0,bounds in definitions:
        starts=[x0.copy(),x0.copy(),x0.copy()]
        starts[1][0]-=1.;starts[2][0]+=1.
        if len(x0)==6:starts[1][1]=.5;starts[2][1]=1.5
        pred=NS[predname]
        def residual_check(theta):
            return np.concatenate([pred(theta,i)-NS['EXP_Y'][:,i] for i in train])
        for index,init in enumerate(starts):
            candidate=least_squares(residual_check,init,bounds=bounds,max_nfev=20000,ftol=1e-11,xtol=1e-11,gtol=1e-11)
            reference=fits[(fold,label)]
            delta=float(candidate.fun@candidate.fun-reference.fun@reference.fun)
            start_audit.append(dict(field_kV_cm=float(NS['EXP_FIELDS_KV_CM'][fold]),model=label,start=index,rss_difference=delta,success=bool(candidate.success)))
assert all(r['success'] for r in start_audit)
assert max(abs(r['rss_difference']) for r in start_audit)<1e-6
(OUT/'original_model_start_audit.json').write_text(json.dumps(start_audit,indent=2))
# Compact, explicit mathematical sensitivity checks.
assert all(r['frozen_gap_s_1']>0 for r in aud)
assert all(r['entropy_derivative_s_1']<0 for r in aud)
assert max(r['detailed_balance_log_error'] for r in aud)<1e-12
assert max(r['relative_column_sum_error'] for r in aud)<1e-12
assert max(r['relative_stationarity_error'] for r in aud)<1e-12
assert np.isclose(fits[(3,'Distributed relaxation')].x[1],chi0,atol=0,rtol=0)
checks=dict(original_model_start_max_rss_difference=max(abs(r['rss_difference']) for r in start_audit),
  max_detailed_balance_log_error=max(r['detailed_balance_log_error'] for r in aud),
  max_relative_column_sum_error=max(r['relative_column_sum_error'] for r in aud),
  all_frozen_gaps_positive=True,all_probe_entropy_derivatives_negative=True,
  nominal_transport_generator_reproduced_exactly=True,
  original_relaxation_parameters_unchanged=True)
(OUT/'revision_checks.json').write_text(json.dumps(checks,indent=2))
print('\nREVISION CHECKS\n',json.dumps(checks,indent=2))


fold=0 Single exponential: train=0.154964201 test=0.164124512 nfev=15


fold=0 Stretched exponential: train=0.0451689771 test=0.0855750747 nfev=8


fold=0 Distributed relaxation: train=0.0273152069 test=0.0840469203 nfev=16


fold=1 Single exponential: train=0.159171682 test=0.151466559 nfev=15


fold=1 Stretched exponential: train=0.0621488302 test=0.0432708307 nfev=7


...

fold=1 Biexponential (six parameters): train=0.0567593959 test=0.0485159742 nfev=19


fold=2 Biexponential (six parameters): train=0.0565062369 test=0.049251298 nfev=15


fold=3 Biexponential (six parameters): train=0.0593903437 test=0.0378099641 nfev=24



CROSS VALIDATION
  field_kV_cm                          model     RMSE       R2
       133.0             Single exponential 0.164125 0.614345
       133.0          Stretched exponential 0.085575 0.895155
       133.0         Distributed relaxation 0.084047 0.898866
       267.0             Single exponential 0.151467 0.695299
       267.0          Stretched exponential 0.043271 0.975133
       267.0         Distributed relaxation 0.040258 0.978475
       400.0             Single exponential 0.153088 0.695322
       400.0          Stretched exponential 0.045089 0.973570
       400.0         Distributed relaxation 0.031322 0.987246
       533.0             Single exponential 0.161174 0.655695
       533.0          Stretched exponential 0.049952 0.966928
       533.0         Distributed relaxation 0.026337 0.990806
       133.0 Biexponential (six parameters) 0.079752 0.908937
       267.0 Biexponential (six parameters) 0.048516 0.968739
       400.0 Biexponential (six parameters) 0.04925


REVISION CHECKS
 {
  "original_model_start_max_rss_difference": 2.641656848823004e-08,
  "max_detailed_balance_log_error": 3.0531133177191805e-16,
  "max_relative_column_sum_error": 1.0100719358196001e-16,
  "all_frozen_gaps_positive": true,
  "all_probe_entropy_derivatives_negative": true,
  "nominal_transport_generator_reproduced_exactly": true,
  "original_relaxation_parameters_unchanged": true
}
